In [ ]:
import torch # install GPU-compatible
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, get_linear_schedule_with_warmup, AutoConfig
from datasets import Dataset
from tqdm import tqdm
from torch.nn import functional as F
import numpy as np
import pandas as pd
from torch.optim import AdamW
from sysml_utils import SysMLASTManager

In [ ]:
parser = SysMLASTManager()

In [ ]:
model_name = "Salesforce/codet5-small"
cache_dir = "./models" 

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
print(config)

In [ ]:
from huggingface_hub import hf_hub_download
import pprint
import json

path = hf_hub_download("Salesforce/codet5p-2b", "config.json")
with open(path) as f:
    cfg = json.load(f)
pprint.pprint(cfg)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, cache_dir=cache_dir)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
basic_df = pd.read_json("dataset/synthetic_dataset_basic_expanded_final.jsonl", lines=True)
domain_df = pd.read_json("dataset/synthetic_dataset_domain_aware_full.jsonl", lines=True) 

merged_df = pd.concat([basic_df, domain_df], ignore_index=True)
dataset = Dataset.from_pandas(merged_df)

In [ ]:
counts = basic_df['source_id'].value_counts().reset_index()
counts.columns = ['source_id', 'num_examples']

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(counts['source_id'], counts['num_examples'])
plt.xlabel('source_id')
plt.ylabel('Number of examples')
plt.title('Examples per source_id')
plt.tight_layout()
plt.show()

counts = domain_df['source_id'].value_counts().reset_index()
counts.columns = ['source_id', 'num_examples']

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(counts['source_id'], counts['num_examples'])
plt.xlabel('source_id')
plt.ylabel('Number of examples')
plt.title('Examples per source_id')
plt.tight_layout()
plt.show()

In [ ]:
# Split the dataset
def group_split(dataset, test_size=0.1, val_size=0.1, seed=42):
    base_ids = list(set(dataset["source_id"]))
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(base_ids)

    n = len(shuffled)
    n_test = int(test_size * n)
    n_val = int(val_size * n)

    test_groups = set(shuffled[:n_test])
    val_groups  = set(shuffled[n_test:n_test+n_val])

    def label_row(example):
        if example["source_id"] in test_groups:
            return {"split": "test"}
        elif example["source_id"] in val_groups:
            return {"split": "val"}
        else:
            return {"split": "train"}

    dataset = dataset.map(label_row)

    train_ds = dataset.filter(lambda x: x["split"] == "train").remove_columns("split")
    val_ds   = dataset.filter(lambda x: x["split"] == "val").remove_columns("split")
    test_ds  = dataset.filter(lambda x: x["split"] == "test").remove_columns("split")

    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = group_split(dataset)
print(f"Datatset Sizes: \nTraining: {len(train_ds)}, Validation: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
### Preprocess and Tokenize the datasets
def preprocess_and_tokenize(batch):
    
    input_texts = [
        f"Fix the following SysML code.\n### Faulty Code:\n{wc}\n### Error:\n{err}"
        for wc, err in zip(batch["bad_code"], batch["error_message"])
    ]

    target_texts = [
        f"Here is the correct code:\n{code}"
        for code in batch["good_code"]
    ]

    inputs = tokenizer(input_texts, truncation=False)
    labels = tokenizer(target_texts, truncation=False)
    
    inputs["labels"] = labels["input_ids"]

    return inputs

train_ds = train_ds.map(preprocess_and_tokenize, batched=True)
val_ds   = val_ds.map(preprocess_and_tokenize, batched=True)
test_ds  = test_ds.map(preprocess_and_tokenize, batched=True)

In [ ]:
print(json.dumps(train_ds[0], indent=2))

In [ ]:
# Truncate over-length token sequences
def check_and_truncate_tokens(
    dataset,
    id_column: str ="id",
    token_columns: list = ["input_ids", "attention_mask", "labels"],
    limit: int = 256,
):
    
    total_truncated = 0
    truncation_details = []

    for idx, example in enumerate(dataset):
        example_id = example.get(id_column, f"row_{idx}")

        for col in token_columns:
            if col not in example:
                raise ValueError(f"Column '{col}' not found in dataset.")
            
            tokens = example[col]
            if not isinstance(tokens, (list, tuple)):
                raise ValueError(f"Column '{col}' should contain a list of token IDs.")

            token_len = len(tokens)

            if token_len > limit:
                truncated_len = token_len - limit
                example[col] = tokens[:limit] 
                truncation_details.append({
                    "index": idx,
                    "id": example_id,
                    "column": col,
                    "original_length": token_len,
                    "truncated_to": limit,
                    "tokens_removed": truncated_len,
                })
                total_truncated += 1

    if total_truncated > 0:
        print(f"Truncated {total_truncated} over-length sequences to {limit} tokens.")
        print("Examples (up to 5 shown):")
        for entry in truncation_details[:5]:
            print(entry)
    else:
        print(f"No truncations needed. All sequences within {limit} tokens.")

    return dataset, {
        "num_truncated": total_truncated,
        "details": truncation_details
    }

train_dataset, train_report = check_and_truncate_tokens(dataset=train_ds)
val_dataset, val_report = check_and_truncate_tokens(dataset=val_ds)
test_dataset, test_report = check_and_truncate_tokens(dataset=test_ds)

In [ ]:
print(train_dataset)

In [ ]:
class CollatorWithText:
    def __init__(self, tokenizer, model):
        self.base_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

    def __call__(self, features):

        token_features = [{k: f[k] for k in f if k in {"input_ids", "attention_mask", "labels"}} 
                          for f in features]

        batch = self.base_collator(token_features)
        return batch

In [ ]:
example = train_dataset[0]
print(example.keys())
for k, v in example.items():
    print(k, type(v))

In [ ]:
### Create DataLoaders
data_collator = CollatorWithText(tokenizer, model)
#data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

def create_dataloader(dataset, batch_size=4, shuffle=True):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=data_collator
    )

train_loader = create_dataloader(train_dataset, batch_size=4, shuffle=True)
val_loader = create_dataloader(val_dataset, batch_size=4, shuffle=False)
test_loader = create_dataloader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
batch = next(iter(train_loader))
list(batch.keys())

for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.shape} | dtype={v.dtype}")
    elif isinstance(v, list):
        print(f"{k}: list of length {len(v)} | first element type: {type(v[0])}")
    else:
        print(f"{k}: {type(v)}")

In [ ]:
def sampled_generation(model, batch):
    model.eval()
    with torch.no_grad():
        preds = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            output_scores=True,
            return_dict_in_generate=True
        )
    model.train()
    return preds


def get_logprobs(outputs):
    all_log_probs = []
    for step, scores in enumerate(outputs.scores):
        log_probs = F.log_softmax(scores, dim=-1)
        next_tokens = outputs.sequences[:, step + 1]
        token_log_probs = log_probs.gather(1, next_tokens.unsqueeze(-1)).squeeze(-1)
        all_log_probs.append(token_log_probs)
        
    log_probs_tensor = torch.stack(all_log_probs, dim=1)
    return log_probs_tensor.sum(dim=1) 


def compute_reinforce_loss(model, batch):
    
    preds = sampled_generation(model, batch)
    log_probs = get_logprobs(preds)

    pred_codes = tokenizer.batch_decode(preds.sequences, skip_special_tokens=True)
    gold_codes = tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)

    rewards = []
    
    for pred, gold in zip(pred_codes, gold_codes):
        pred_ast = parser.code_to_ast(pred)
        gold_ast = parser.code_to_ast(gold)
        
        if isinstance(gold_ast, str):
            raise ValueError("Ground truth code could not be parsed into AST.")
        if isinstance(pred_ast, str):
            rewards.append(-1.0) # Penalty for unparsable code
        elif isinstance(pred_ast, dict):
            similarity = parser.compare_asts(pred_ast, gold_ast)
            rewards.append(similarity)

    rewards = torch.tensor(rewards, dtype=torch.float, device=log_probs.device)
    advantages = rewards - rewards.mean()
    loss = -(advantages * log_probs).mean()
    return loss

In [ ]:
import torch
from tqdm import tqdm
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from transformers import get_linear_schedule_with_warmup

# --- safety setup ---
torch.cuda.empty_cache()
model.gradient_checkpointing_enable()  # saves a lot of VRAM
scaler = GradScaler('cuda')

num_epochs = 10
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

train_losses, val_losses = [], []

def run_epoch(dataloader, evaluating=False):
    running_loss, count = 0.0, 0

    if evaluating:
        model.eval()
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Validation", leave=False):
                batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                running_loss += loss.item()
                count += 1

    else:
        model.train()
        for batch in tqdm(dataloader, desc="Training", leave=False):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            optimizer.zero_grad(set_to_none=True)

            # --- mixed precision (keep, but fix usage) ---
            with autocast(device_type='cuda'):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            count += 1

            # only delete large objects; do NOT empty cache per batch
            del batch, outputs, loss

    # empty cache once per epoch, not every batch
    torch.cuda.empty_cache()

    return running_loss / count if count else 0.0



for epoch in range(num_epochs):
    train_loss = run_epoch(train_loader, evaluating=False)
    val_loss = run_epoch(val_loader, evaluating=True)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    torch.cuda.empty_cache()

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, num_epochs + 1)

plt.figure()
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
# CrossEntropyLoss ignoring padding (-100) and returning per-token loss
loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100, reduction="none")

def compute_ast_similarity(pred_code: str, gold_code: str):
    pass # placeholder #TODO 

def compute_ast_weights(model, labels, batch):
    model.eval()
    ast_weights = []

    preds = model.generate(batch["input_ids"])
    pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
    correct_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    for pred, gold in zip(pred_texts, correct_texts):
        weight = compute_ast_similarity(pred, gold)
        ast_weights.append(weight)

    model.train()

    return ast_weights


def compute_ce_loss(logits, labels):

    # Shift for decoder: drop last logit, drop first label (BOS)
    logits = logits[:, :-1, :]
    labels = labels[:, 1:]

    batch_size = labels.size(0)
    seq_len = labels.size(1)

    # Flatten for CE computation: [B*(T-1), vocab_size] vs [B*(T-1)]
    logits = logits.reshape(-1, logits.size(-1))
    labels = labels.reshape(-1)

    # Compute per-token CE loss
    loss_per_token = loss_fct(logits, labels)

    # Reshape back to [batch, seq_len-1]
    loss_per_token = loss_per_token.reshape(batch_size, seq_len)

    # Average per sequence [batch, 1]
    loss_per_seq = loss_per_token.mean(dim=1)

    return loss_per_seq



def compute_loss(model, outputs, batch):

    logits = outputs.logits  # [batch, seq_len, vocab_size]
    labels = batch["labels"]  # [batch, seq_len]

    ce_loss_per_example = compute_ce_loss(logits, labels)
    ast_weights = compute_ast_weights(model, labels, batch)

    weighted_loss = (ce_loss_per_example * ast_weights)

    return weighted_loss.mean()